# BTCUSDT 1-Minute Data Quality Check

This notebook performs thesis-ready exploratory validation for the processed BTCUSDT 1-minute dataset. It checks schema integrity, missing values, temporal continuity, and key stylized behaviors (price path, returns, volatility, and return distribution).

## 1. Setup and Data Loading

This section resolves project paths, loads processed parquet, and confirms that expected files are available.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / 'src').exists():
    project_root = project_root.parent

processed_path = project_root / 'data' / 'processed' / 'crypto' / '1min' / 'BTCUSDT_1m_processed.parquet'
missing_diag_path = project_root / 'data' / 'metadata' / 'BTCUSDT_1m_missing_timestamps.csv'

print(f'Project root: {project_root}')
print(f'Processed file exists: {processed_path.exists()}')
print(f'Missing diagnostics exists: {missing_diag_path.exists()}')

df = pd.read_parquet(processed_path)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df = df.sort_values('timestamp').reset_index(drop=True)
df.head()

## 2. Structural Summary

We inspect head/tail, data types, missing-value counts, and global date range to validate the processed dataset structure before downstream motif experiments.

In [ ]:
display(df.head(3))
display(df.tail(3))

dtype_summary = df.dtypes.astype(str).rename('dtype').to_frame()
missing_summary = df.isna().sum().rename('missing_count').to_frame()

display(dtype_summary)
display(missing_summary)

date_min = df['timestamp'].min()
date_max = df['timestamp'].max()
row_count = len(df)

print(f'Row count: {row_count:,}')
print(f'Date range (UTC): {date_min} -> {date_max}')

## 3. Price Series Behavior

Plotting close prices helps verify data continuity and identifies obvious anomalies (flatlines, spikes, truncations).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['timestamp'], df['close'], linewidth=0.7, color='tab:blue')
ax.set_title('BTCUSDT Close Price (1m)')
ax.set_xlabel('Timestamp (UTC)')
ax.set_ylabel('Price')
plt.tight_layout()
plt.show()

## 4. Return Dynamics

Log returns should fluctuate around zero with volatility clustering; this plot is a quick sanity check before motif mining and regime conditioning.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['timestamp'], df['log_return'], linewidth=0.6, color='tab:orange')
ax.set_title('BTCUSDT Log Returns (1m)')
ax.set_xlabel('Timestamp (UTC)')
ax.set_ylabel('Log Return')
plt.tight_layout()
plt.show()

## 5. Rolling Volatility Features

We visualize engineered rolling volatility features to verify expected scale and time variation.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['timestamp'], df['volatility_30m'], label='volatility_30m', linewidth=0.7)
ax.plot(df['timestamp'], df['volatility_60m'], label='volatility_60m', linewidth=0.7)
ax.plot(df['timestamp'], df['volatility_240m'], label='volatility_240m', linewidth=0.8)
ax.set_title('Rolling Volatility Features')
ax.set_xlabel('Timestamp (UTC)')
ax.set_ylabel('Volatility')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Return Distribution

A histogram of log returns supports quick checks of heavy tails and potential outliers relevant for motif robustness analysis.

In [ ]:
returns = df['log_return'].dropna()
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(returns, bins=120, color='tab:green', alpha=0.8)
ax.set_title('Distribution of BTCUSDT 1m Log Returns')
ax.set_xlabel('Log Return')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

print('Return summary:')
display(returns.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

## 7. Missing Timestamp Diagnostics

This section checks explicit missing timestamp diagnostics from pipeline metadata and recomputes an in-notebook missing-bar count for cross-validation.

In [ ]:
if missing_diag_path.exists():
    missing_df = pd.read_csv(missing_diag_path)
    print(f'Missing timestamps reported by pipeline: {len(missing_df):,}')
    display(missing_df.head(10))
else:
    print('Missing timestamp diagnostics CSV not found.')

expected_range = pd.date_range(df['timestamp'].min(), df['timestamp'].max(), freq='1min', tz='UTC')
observed = pd.DatetimeIndex(df['timestamp'])
missing_recomputed = expected_range.difference(observed)

print(f'Recomputed missing timestamps: {len(missing_recomputed):,}')
if len(missing_recomputed) > 0:
    display(pd.DataFrame({'timestamp': missing_recomputed[:10]}))

## 8. Interpretation Notes for Thesis Workflow

- Initial NaNs in return/volatility columns are expected from differencing and rolling windows.
- Any non-zero missing-bar count should be handled explicitly before motif benchmarking.
- These diagnostics should be rerun after every raw data refresh to preserve reproducibility.